# RSNA Knee Abnormality: Exploratory Data Analysis

This notebook checks the study count, the 12 available label counts, series metadata coverage, and the report-only versus explicitly labeled split.

In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path('..') / 'data'
train = pd.read_csv(DATA_DIR / 'train.csv')
train_series = pd.read_csv(DATA_DIR / 'train_series.csv')

print(f'train.csv shape: {train.shape}')
print(f'train_series.csv shape: {train_series.shape}')
display(train.head(2))
display(train_series.head(2))

In [ ]:
label_columns = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus',
    'Medial OA', 'Lateral OA', 'PF OA', 'Effusion',
    'Synovitis', "Baker's", 'Contusion', 'Fracture'
]
missing_labels = sorted(set(label_columns) - set(train.columns))
if missing_labels:
    raise KeyError(f'Missing expected label columns: {missing_labels}')

has_any_label = train[label_columns].notna().any(axis=1)
has_report = train['Report'].fillna('').str.strip().ne('')
report_only = has_report & ~has_any_label

print(f"Number of unique studies: {train['StudyInstanceUID'].nunique():,}")
print(f"Studies with at least one explicit label: {has_any_label.sum():,}")
print(f"Studies with a non-empty report but no explicit labels: {report_only.sum():,}")
print('Non-null rows for each label:')
display(train[label_columns].notna().sum().rename('studies_with_label').to_frame())

In [ ]:
print(f"Unique studies represented in train_series.csv: {train_series['StudyInstanceUID'].nunique():,}")

metadata_columns = [
    column for column in train_series.columns
    if any(term in column.lower() for term in ('scanner', 'site', 'hospital', 'manufacturer', 'institution'))
]
if metadata_columns:
    print('Scanner/site metadata columns found:')
    for column in metadata_columns:
        print(f"  {column}: {train_series[column].nunique(dropna=True):,} unique values")
else:
    print('No scanner/site columns are present in train_series.csv.')
    print('Available series metadata:', ', '.join(train_series.columns))

print('Split summary:')
print(f"  Explicitly labeled studies: {has_any_label.sum():,}")
print(f"  Report-derived studies (report, no explicit labels): {report_only.sum():,}")
print(f"  Studies with neither: {(~has_report & ~has_any_label).sum():,}")

## Grouped cross-validation folds

The downloaded series metadata has no manufacturer, institution, site, or scanner field. The fallback groups by `StudyInstanceUID`, preventing study leakage while preserving the limitation that these folds are not site-held-out folds.

In [ ]:
from sklearn.model_selection import GroupKFold

site_terms = ('scanner', 'site', 'hospital', 'manufacturer', 'institution')
site_columns = [
    column for column in train_series.columns
    if any(term in column.lower() for term in site_terms)
]

if site_columns:
    group_column = site_columns[0]
    site_by_study = train_series.groupby('StudyInstanceUID')[group_column].nunique(dropna=True)
    if (site_by_study > 1).any():
        raise ValueError(f'{group_column} has multiple values within a study.')
    group_by_study = train_series.groupby('StudyInstanceUID')[group_column].first()
    train['cv_group'] = train['StudyInstanceUID'].map(group_by_study).fillna('missing_site')
    print(f'Grouping by metadata column: {group_column}')
else:
    train['cv_group'] = train['StudyInstanceUID']
    print('No site/scanner metadata found; grouping by StudyInstanceUID instead.')

n_splits = 5
if train['cv_group'].nunique() < n_splits:
    raise ValueError('There are fewer groups than requested cross-validation folds.')

group_kfold = GroupKFold(n_splits=n_splits)
train['fold'] = -1
for fold_number, (_, validation_indices) in enumerate(
    group_kfold.split(train, groups=train['cv_group'])
):
    train.loc[train.index[validation_indices], 'fold'] = fold_number

folds = train[['StudyInstanceUID', 'fold']].copy()
folds.to_csv(DATA_DIR / 'folds.csv', index=False)

print(f'Groups: {train["cv_group"].nunique():,}')
print(folds['fold'].value_counts().sort_index().rename('rows'))
print(f'Wrote {len(folds):,} assignments to {DATA_DIR / "folds.csv"}')